First we need to download and extract the data, which can be done with the wget and unzip commands:

In [75]:
# https://archive.ics.uci.edu/ml/machine-learning-databases/00462/drugsCom_raw.zip
# !unzip drugsCom_raw.zip

Since TSV is just a variant of CSV that uses tabs instead of commas as the separator, we can load these files by using the csv loading script and specifying the delimiter argument in the load_dataset() function as follows:

In [76]:
import numpy as np 
import pandas as pd

train = pd.read_csv("drugsComTrain_raw.tsv" , delimiter = "\t")

test = pd.read_csv("drugsComTest_raw.tsv", delimiter = "\t")

In [77]:
train = train.sample(1000)

In [78]:
train.shape

(1000, 7)

In [79]:
test = test.sample(1000)

In [80]:
test

,Unnamed: 0,drugName,condition,review,rating,date,usefulCount
7692,214044,Tioconazole,Vaginal Yeast Infection,"""I&#039;ve suffered from YIs on and off for ye...",2.0,"July 3, 2016",5
42378,199689,Varenicline,Smoking Cessation,"""Thank you so much for Chantix! I have bullou...",10.0,"March 13, 2017",36
2622,95816,Sertraline,Post Traumatic Stress Disorde,"""49 yo female with post traumatic stress disor...",8.0,"April 29, 2014",13
2849,116554,Advil,Pain,"""I love this product! I was in severe pain unt...",10.0,"May 22, 2008",21
4535,29919,Klonopin,Panic Disorde,"""I was diagnosed as early as nine with GAD as ...",10.0,"September 26, 2017",13
...,...,...,...,...,...,...,...
26677,117977,Imuran,Ulcerative Colitis,"""With Imuran, I had no quality of life. Daily ...",1.0,"February 26, 2017",1
5816,4145,Atropine / diphenoxylate,Diarrhea,"""Works so fast. Best help I ever had to contro...",10.0,"August 4, 2009",69
40391,61923,Citalopram,Anxiety and Stress,"""This medication did not help me. I took it th...",1.0,"July 28, 2015",15
9379,203797,Suvorexant,Insomnia,"""Worked very well. Did have next day drowsines...",8.0,"November 20, 2016",27


In [52]:
train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1000 entries, 62322 to 12525
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Unnamed: 0   1000 non-null   int64  
 1   drugName     1000 non-null   object 
 2   condition    993 non-null    object 
 3   review       1000 non-null   object 
 4   rating       1000 non-null   float64
 5   date         1000 non-null   object 
 6   usefulCount  1000 non-null   int64  
dtypes: float64(1), int64(2), object(4)
memory usage: 62.5+ KB


Let’s see how we can use Datasets to deal with each of these issues. To test the patient ID hypothesis for the Unnamed: 0 column, we can use the unique() function to verify that the number of IDs matches the number of rows in each split:

In [53]:
train["Unnamed: 0"].duplicated().count().sum()

np.int64(1000)

In [54]:
test["Unnamed: 0"].duplicated().count()

np.int64(1000)

This seems to confirm our hypothesis, so let’s clean up the dataset a bit by renaming the Unnamed: 0 column to something a bit more interpretable. We can use the rename_column() function to rename the column across both splits in one go:

In [62]:
train= train.rename(columns={"Unnamed: 0": "patient_id"})

In [63]:
train

,patient_id,drugName,condition,review,rating,date,usefulCount
62322,131574,Effexor XR,Generalized Anxiety Disorde,"""I started taking Effexor around 3 years ago, ...",4.0,"August 9, 2017",6
102822,168886,Vilazodone,Depression,"""Works very well but there are pretty severe w...",8.0,"March 14, 2013",51
131130,173074,Clonazepam,Anxiety,"""I just started taking the Klonopin yesterday....",9.0,"February 4, 2011",11
106105,72796,Ethinyl estradiol / norethindrone,Birth Control,"""Man. It hasn&#039;t been a month &amp; my bod...",5.0,"July 22, 2015",3
31841,96654,Neurontin,Reflex Sympathetic Dystrophy Syndrome,"""Have stage 3 RSD. This medicine has helped ma...",8.0,"October 28, 2009",21
...,...,...,...,...,...,...,...
133537,110782,Imodium,Diarrhea,"""Works.""",9.0,"March 11, 2014",10
141063,229333,Lurasidone,Bipolar Disorde,"""I have been on Latuda 40mg for 3 weeks now. I...",9.0,"July 22, 2017",1
137582,44611,Ethinyl estradiol / norgestimate,Birth Control,"""I have been using Ortho Tri-Cyclen Lo for 6 y...",10.0,"September 3, 2009",59
31930,184712,Canagliflozin,"Diabetes, Type 2","""I have been taking Invokana 100mg for 1 yr...",10.0,"January 3, 2017",33


Next, let’s normalize all the condition labels using Dataset.map()

In [ ]:
def lower_case(example):
    return {"condition": example["condition"].lower()}

train = train.map(lower_case)

 Let’s drop these rows using Dataset.filter(), which works in a similar way to Dataset.map() and expects a function that receives a single example of the dataset. Instead of writing an explicit function like:

In [ ]:
def filter_nones(x):
    return x["condition"] is not None

In [ ]:
train = train.filter(lambda x: x["condition"] is not None)

In [ ]:
train = train.map(lower_case)

In [83]:
import numpy as np
import pandas as pd

train = pd.read_csv("drugsComTrain_raw.tsv", delimiter="\t")
test = pd.read_csv("drugsComTest_raw.tsv", delimiter="\t")

train_small = train.sample(5000, random_state=42)
test_small = test.sample(5000, random_state=42)

train_small.to_csv("train_5000.tsv", sep="\t", index=False)
test_small.to_csv("test_5000.tsv", sep="\t", index=False)